In [1]:
import pyarrow.parquet as pq
import os
import duckdb

In [2]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line"
output_dir = os.path.join(base_path, "filtered")

materials_file = os.path.join(base_path, "materials_single_line.parquet")
filtered_columns_file = os.path.join(output_dir, "filtered_materials.parquet")
final_output_file = os.path.join(output_dir, "filtered_materials_encoded_time_cut.parquet")

## Step 1: Filter columns with PyArrow

In [3]:
# Columns to retain
keep_columns = [
    "component_position", "component_id", "serial_number_id",
    "station_id", "supplier_id", "mounting_place",
    "container_number", "panel_position", "created_at"
]

In [4]:
# Load the full Parquet file
table = pq.read_table(materials_file)

# Drop columns not in the keep list
columns_to_drop = [col for col in table.column_names if col not in keep_columns]
filtered_table = table.drop(columns_to_drop)

In [5]:
# Save the filtered table first (without encoding yet)
pq.write_table(filtered_table, filtered_columns_file)
print(f"Filtered materials saved to: {filtered_columns_file}")

Filtered materials saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_materials.parquet


 ## Step 2: Encode the container_number column, fill in mounting_place null values and cut out only the desired time frame

In [3]:
con = duckdb.connect()

Define target time window, this is the timewindow of the bookings and measurements merged dataset.

In [4]:
# TODO Adjust or automate to match the timeframe of bookings and measurements
start_time = "2025-03-01T01:21:20.773Z"
end_time = "2025-05-14T01:02:06.180Z"

In [5]:
# Load the parquet, fill NULL mounting_place, frequency encode container_number
con.execute(f"""
CREATE OR REPLACE TABLE materials AS
SELECT *,
       COUNT(*) OVER (PARTITION BY container_number) AS container_number_freq
FROM (
    SELECT
        component_position, component_id, serial_number_id,
        station_id, supplier_id,
        COALESCE(mounting_place, 'unknown') AS mounting_place,
        container_number, panel_position, created_at
    FROM '{filtered_columns_file}'
);
""")

In [6]:
# Filter by time window
con.execute("""
CREATE OR REPLACE TABLE filtered_materials_final AS
SELECT
    component_position, component_id, serial_number_id,
    station_id, supplier_id, mounting_place,
    panel_position, created_at, container_number_freq
FROM materials
WHERE created_at BETWEEN ? AND ?;
""", [start_time, end_time])

# Export to parquet
con.execute(f"COPY filtered_materials_final TO '{final_output_file}' (FORMAT 'parquet');")
print(f"✅ Final filtered, encoded, and cleaned materials saved to: {final_output_file}")

✅ Final filtered, encoded, and cleaned materials saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_materials_encoded_time_cut.parquet


In [7]:
con.close()